# FastAPI Endpoints Test Notebook

This notebook tests the following FastAPI endpoints:
- `/api/v1/documents/upload` - Upload documents
- `/api/v1/documents/{id}/download` - Download documents
- `/api/v1/search/` - Search documents
- `/api/v1/search/semantic` - Semantic search
- `/api/v1/analytics/overview` - Analytics
- `/api/v1/tags/` - List tags

Test files are located in: `/Users/glennmossy/dpg-ai-projects/claude_document_mcp_server/testfiles`


In [ ]:
import requests
import json
from pathlib import Path
from typing import Dict, Any
import os

# API base URL - adjust if needed
BASE_URL = "http://localhost/api/v1"  # Using nginx reverse proxy
# BASE_URL = "http://localhost:8000/api/v1"  # Direct backend access

# Test files directory
TESTFILES_DIR = Path("/Users/glennmossy/dpg-ai-projects/claude_document_mcp_server/testfiles")

print(f"API Base URL: {BASE_URL}")
print(f"Test Files Directory: {TESTFILES_DIR}")
print(f"Test Files Directory Exists: {TESTFILES_DIR.exists()}")


## 1. Health Check


In [ ]:
# Health check
response = requests.get(f"{BASE_URL}/healthz")
print(f"Status: {response.status_code}")
print(f"Response: {response.json()}")


## 2. Upload Document Test


In [ ]:
# List available test files
test_files = list(TESTFILES_DIR.glob("*")) if TESTFILES_DIR.exists() else []
print(f"Available test files: {len(test_files)}")
for f in test_files[:10]:  # Show first 10
    print(f"  - {f.name} ({f.stat().st_size} bytes)")

# Upload a test file
uploaded_documents = []

if test_files:
    test_file = test_files[0]  # Use first available file
    
    with open(test_file, 'rb') as f:
        files = {'file': (test_file.name, f, 'application/octet-stream')}
        data = {
            'title': f'Test Document - {test_file.name}',
            'tags': json.dumps(['test', 'notebook', 'api']),
            'status': 'draft',
            'metadata': json.dumps({
                'source': 'test_notebook',
                'category': 'testing'
            })
        }
        
        response = requests.post(f"{BASE_URL}/documents/upload", files=files, data=data)
        
        print(f"\nUpload Status: {response.status_code}")
        if response.status_code == 200:
            result = response.json()
            print(f"\nUpload Response:")
            print(json.dumps(result, indent=2))
            
            # Store document ID for later tests
            if 'document_id' in result:
                uploaded_documents.append({
                    'document_id': result['document_id'],
                    'title': result.get('title', ''),
                    'version': result.get('version', 1)
                })
                print(f"\n✅ Document uploaded successfully!")
                print(f"   Document ID: {result['document_id']}")
                print(f"   Version: {result.get('version', 1)}")
        else:
            print(f"❌ Upload failed: {response.text}")
else:
    print("⚠️  No test files found in testfiles directory")


## 3. Download Document Test


In [ ]:
# Download the uploaded document
if uploaded_documents:
    doc = uploaded_documents[0]
    doc_id = doc['document_id']
    
    response = requests.get(f"{BASE_URL}/documents/{doc_id}/download")
    
    print(f"Download Status: {response.status_code}")
    
    if response.status_code == 200:
        # Get filename from Content-Disposition header
        content_disposition = response.headers.get('Content-Disposition', '')
        print(f"Content-Disposition: {content_disposition}")
        print(f"Content-Type: {response.headers.get('Content-Type', 'unknown')}")
        print(f"Content-Length: {len(response.content)} bytes")
        
        # Save downloaded file
        output_file = TESTFILES_DIR / f"downloaded_{doc_id}.bin"
        with open(output_file, 'wb') as f:
            f.write(response.content)
        
        print(f"\n✅ Document downloaded successfully!")
        print(f"   Saved to: {output_file}")
    else:
        print(f"❌ Download failed: {response.text}")
else:
    print("⚠️  No documents uploaded yet. Run the upload test first.")


## 4. Search Documents Test


In [ ]:
# Search for documents
search_query = "test"
limit = 10

response = requests.get(f"{BASE_URL}/search/", params={'q': search_query, 'limit': limit})

print(f"Search Status: {response.status_code}")

if response.status_code == 200:
    result = response.json()
    print(f"\nSearch Results for '{search_query}':")
    print(json.dumps(result, indent=2))
    
    results = result.get('results', [])
    print(f"\n✅ Found {len(results)} documents")
    
    for doc in results[:5]:  # Show first 5
        print(f"  - {doc.get('title', 'N/A')} (ID: {doc.get('document_id', 'N/A')})")
else:
    print(f"❌ Search failed: {response.text}")


## 5. Semantic Search Test


In [ ]:
# Semantic search
semantic_query = "test document"
limit = 10

payload = {
    'query': semantic_query,
    'limit': limit
}

response = requests.post(f"{BASE_URL}/search/semantic", json=payload)

print(f"Semantic Search Status: {response.status_code}")

if response.status_code == 200:
    result = response.json()
    print(f"\nSemantic Search Results for '{semantic_query}':")
    print(json.dumps(result, indent=2))
    
    results = result.get('results', [])
    print(f"\n✅ Found {len(results)} documents")
else:
    print(f"❌ Semantic search failed: {response.text}")
    print(f"Note: Semantic search may not be fully implemented")


## 6. Analytics Overview Test


In [ ]:
# Get analytics overview
response = requests.get(f"{BASE_URL}/analytics/overview")

print(f"Analytics Status: {response.status_code}")

if response.status_code == 200:
    result = response.json()
    print(f"\nAnalytics Overview:")
    print(json.dumps(result, indent=2))
    
    totals = result.get('totals', {})
    print(f"\n✅ Analytics retrieved successfully")
    print(f"   Totals: {totals}")
else:
    print(f"❌ Analytics request failed: {response.text}")


## 7. List Tags Test


In [ ]:
# List all tags
response = requests.get(f"{BASE_URL}/tags/", params={
    'sort_by_count': True,
    'min_count': 1
})

print(f"Tags Status: {response.status_code}")

if response.status_code == 200:
    result = response.json()
    print(f"\nTags List:")
    print(json.dumps(result, indent=2))
    
    tags = result.get('tags', [])
    total = result.get('total', 0)
    
    print(f"\n✅ Found {total} unique tags")
    
    for tag_info in tags[:10]:  # Show first 10
        tag_name = tag_info.get('tag', 'N/A')
        count = tag_info.get('count', 0)
        print(f"  - {tag_name}: {count} documents")
else:
    print(f"❌ Tags request failed: {response.text}")


## 8. Test Summary


In [ ]:
# Summary of uploaded documents
print("\n" + "="*50)
print("TEST SUMMARY")
print("="*50)

print(f"\nUploaded Documents: {len(uploaded_documents)}")
for doc in uploaded_documents:
    print(f"  - {doc['title']} (ID: {doc['document_id']}, Version: {doc['version']})")

print(f"\n✅ All endpoint tests completed!")
print(f"\nTest files directory: {TESTFILES_DIR}")
print(f"API Base URL: {BASE_URL}")
